# INE/MINVU Áreas Verdes – Exploración y superficies de plazas/parques por comuna

Este notebook:
- Verifica el entorno/KERNEL
- Inspecciona metadata y campos del FeatureServer
- Extrae valores (plaza/parque) y calcula superficies agregadas por comuna (m²)
- Aplica QA/QC para llaves (CUT) y consolida nombres de comuna
- Exporta a CSV

**Fuente de datos:** servicio ArcGIS `Áreas_Verdes_MINVU_INE_2023` (FeatureServer).


In [1]:
# --- 0) Setup ---
import sys
import json
import requests
import pandas as pd
from pprint import pprint

print("Python:", sys.version)
print("Executable:", sys.executable)
print("pandas:", pd.__version__)
print("requests:", requests.__version__)


Python: 3.11.6 | packaged by conda-forge | (main, Oct  3 2023, 11:57:02) [GCC 12.3.0]
Executable: /opt/conda/bin/python
pandas: 2.1.1
requests: 2.31.0


## 1) Configuración del servicio ArcGIS
Usamos la capa 0 del FeatureServer.

In [2]:
layer_url = "https://geoide.minvu.cl/server/rest/services/Catastros/%C3%81reas_Verdes_MINVU_INE_2023/FeatureServer/0"
query_url = layer_url + "/query"

# Helper robusto: devuelve DataFrame y muestra errores ArcGIS si existen
def arcgis_query(params, timeout=90):
    params = {**params, "f": "json"}
    r = requests.get(query_url, params=params, timeout=timeout).json()
    if "error" in r:
        print("ArcGIS error:")
        pprint(r["error"])
        return pd.DataFrame(), r
    feats = r.get("features", [])
    df = pd.DataFrame([f.get("attributes", {}) for f in feats])
    return df, r

def arcgis_meta(timeout=60):
    r = requests.get(layer_url, params={"f": "pjson"}, timeout=timeout).json()
    return r

meta = arcgis_meta()
print("Layer name:", meta.get("name"))
print("Geometry type:", meta.get("geometryType"))
print("ObjectID field:", meta.get("objectIdField"))


Layer name: Areas_Verdes_MINVU_INE_2023
Geometry type: esriGeometryPolygon
ObjectID field: OBJECTID_1


## 2) Listar campos ("nombres de columnas") y buscar variables de superficie/fecha
Buscamos campos típicos de superficie y también si existe algún campo de fecha/año (en este layer no suele existir).

In [3]:
fields = meta.get("fields", [])
field_names = [f["name"] for f in fields]
print("N° campos:", len(field_names))
print("Primeros 40 campos:", field_names[:40])

# Campos de superficie / área
area_candidates = [c for c in field_names if any(k in c.upper() for k in ["SUP", "AREA", "M2"])]
print("\nCandidatos de superficie/area:", area_candidates)

# Campos de fecha/año (creación real, edición, etc.)
date_candidates = [c for c in field_names if any(k in c.upper() for k in ["ANIO", "AÑO", "YEAR", "FECHA", "DATE", "CREAT", "EDIT", "INAG", "APERT", "CONSTR"])]
print("Candidatos de fecha/año:", date_candidates)


N° campos: 20
Primeros 40 campos: ['OBJECTID_1', 'OBJECTID', 'CUT', 'REGION', 'PROVINCIA', 'COMUNA', 'AREA_C', 'COD_URBANO', 'TIPO_EP', 'NOMBRE_EP', 'OBSERVACIO', 'SUP_TOTAL_', 'SUP_VEGETA', 'GRUPO_URB', 'USO', 'TIPO_TAMAN', 'ID_AV', 'ADM', 'Shape__Area', 'Shape__Length']

Candidatos de superficie/area: ['AREA_C', 'SUP_TOTAL_', 'SUP_VEGETA', 'Shape__Area']
Candidatos de fecha/año: []


## 3) Ver valores de clasificación (PLAZA/PARQUE) y USO
Muestreamos registros y sacamos valores únicos (evita depender de `returnDistinctValues`).

In [4]:
df_sample, raw = arcgis_query({
    "where": "1=1",
    "outFields": "AREA_C,TIPO_EP,USO",
    "returnGeometry": "false",
    "resultRecordCount": 5000
})

print("Registros muestreados:", len(df_sample))
display(df_sample.head())

for col in ["AREA_C", "TIPO_EP", "USO"]:
    if col in df_sample.columns:
        print("\nValores únicos (top 20) para", col)
        print(df_sample[col].fillna("<NA>").astype(str).str.strip().value_counts().head(20))


Registros muestreados: 2000


,AREA_C,TIPO_EP,USO
0,URBANO,PLAZA,PÚBLICO
1,URBANO,PLAZA,PÚBLICO
2,URBANO,PLAZA,PÚBLICO
3,URBANO,PLAZA,PÚBLICO
4,URBANO,PLAZA,PÚBLICO



Valores únicos (top 20) para AREA_C
AREA_C
URBANO    1998
MIXTO        2
Name: count, dtype: int64

Valores únicos (top 20) para TIPO_EP
TIPO_EP
PLAZA     1879
PARQUE     121
Name: count, dtype: int64

Valores únicos (top 20) para USO
USO
PÚBLICO    2000
Name: count, dtype: int64


## 4) Definir campo de superficie
Para este servicio, un buen campo es `SUP_TOTAL_` (m²).

In [5]:
AREA_FIELD = "SUP_TOTAL_"
assert AREA_FIELD in field_names, f"No existe el campo {AREA_FIELD}. Revisa area_candidates arriba."
print("Campo de superficie usado:", AREA_FIELD)


Campo de superficie usado: SUP_TOTAL_


## 5) Calcular superficies agregadas por comuna (m²)
Usamos `outStatistics` para sumar `SUP_TOTAL_` por CUT y COMUNA, filtrando por tipo y uso público.

In [6]:
def superficie_por_comuna(tipo_ep, uso="PÚBLICO", area_field=AREA_FIELD):
    where = f"TIPO_EP = '{tipo_ep}' AND USO = '{uso}'"
    df, raw = arcgis_query({
        "where": where,
        "groupByFieldsForStatistics": "CUT,COMUNA",
        "outStatistics": f'[{{"statisticType":"sum","onStatisticField":"{area_field}","outStatisticFieldName":"SUP_M2"}}]',
        "outFields": "CUT,COMUNA",
        "returnGeometry": "false"
    })
    return df

plazas = superficie_por_comuna("PLAZA")
parques = superficie_por_comuna("PARQUE")

print("Comunas con plazas:", len(plazas))
print("Comunas con parques:", len(parques))
display(plazas.head())
display(parques.head())


Comunas con plazas: 87
Comunas con parques: 115


,CUT,COMUNA,SUP_M2
0,2201,CALAMA,449813.188260
1,13130,SAN MIGUEL,100097.066637
2,13123,PROVIDENCIA,175191.514248
3,5601,SAN ANTONIO,175850.567593
4,5109,VIÑA DEL MAR,278638.088248


,CUT,COMUNA,SUP_M2
0,11201,Aisén,25343.533287
1,13502,Alhué,14173.570419
2,1107,ALTO HOSPICIO,143979.219425
3,9201,Angol,59847.731353
4,2101,Antofagasta,441060.447419


## 6) QA/QC de llaves y consolidación
El merge debe hacerse por **CUT** (clave oficial). Normalizamos nombres y consolidamos una sola columna COMUNA.

In [7]:
def normaliza_texto(s: pd.Series) -> pd.Series:
    return (s.astype(str)
              .str.strip()
              .str.upper())

for df in [plazas, parques]:
    df["CUT"] = pd.to_numeric(df["CUT"], errors="coerce").astype("Int64")
    df["COMUNA"] = normaliza_texto(df["COMUNA"])

# Renombrar métricas
plazas = plazas.rename(columns={"SUP_M2": "SUP_PLAZAS_M2"})
parques = parques.rename(columns={"SUP_M2": "SUP_PARQUES_M2"})

# Merge por CUT (evita duplicaciones por diferencias en COMUNA)
out = plazas.merge(parques, on="CUT", how="outer", suffixes=("_plazas", "_parques"))

# Consolidar nombre de comuna (si viene de ambos lados)
if "COMUNA_plazas" in out.columns and "COMUNA_parques" in out.columns:
    out["COMUNA"] = out["COMUNA_plazas"].combine_first(out["COMUNA_parques"])
    out = out.drop(columns=["COMUNA_plazas", "COMUNA_parques"])

# Orden y tipos
out = out[["CUT", "COMUNA", "SUP_PLAZAS_M2", "SUP_PARQUES_M2"]].sort_values("CUT")

display(out.head(20))

# Checks
print("\nChecks:")
print("Filas (comunas):", len(out))
print("N CUT duplicados:", out["CUT"].duplicated().sum())
print("Total m² plazas:", out["SUP_PLAZAS_M2"].fillna(0).sum())
print("Total m² parques:", out["SUP_PARQUES_M2"].fillna(0).sum())


,CUT,COMUNA,SUP_PLAZAS_M2,SUP_PARQUES_M2
0,1101,IQUIQUE,183909.065543,2.464326e+05
1,1107,ALTO HOSPICIO,136082.365444,1.439792e+05
2,2101,ANTOFAGASTA,369628.160978,4.410604e+05
3,2104,TALTAL,NaN,3.618286e+04
4,2201,CALAMA,449813.188260,3.235875e+05
5,3101,COPIAPÓ,431643.236638,3.701818e+05
6,3102,CALDERA,NaN,4.907928e+04
7,3202,DIEGO DE ALMAGRO,NaN,8.045413e+04
8,4101,LA SERENA,781566.469949,1.569069e+06
9,4102,COQUIMBO,679009.706627,2.627900e+05



Checks:
Filas (comunas): 118
N CUT duplicados: 1
Total m² plazas: 30753822.58842784
Total m² parques: 44114928.708357446


## 7) Exportar a CSV
Guarda un archivo con superficies por comuna.

In [8]:
out.to_csv("superficie_plazas_parques_por_comuna.csv", index=False)
print("Guardado:", "superficie_plazas_parques_por_comuna.csv")


Guardado: superficie_plazas_parques_por_comuna.csv


## 8) Nota sobre 'año de creación real'
Este dataset no incluye un atributo del tipo `ANIO_CREACION`/`FECHA_INAUG` por parque/plaza. Si necesitas año de creación real, suele requerir fuentes administrativas (municipales/programas) o inferencia por comparación de catastros históricos.